In [9]:
def volume_control(input, output, gain):
    for i in range(1024):
        output[i] = input[i] * gain

In [10]:
def echo_control(input_block, output_block, delay_buf, decay):
    for i in range(1024):
        output_block[i] = input_block[i] + delay_buf[i] * decay
        delay_buf[i] = input_block[i]

In [41]:
import numpy as np

def pitch_shift(input, tones, N=1024, H=256):

    ratio = 2 ** (tones/12)

    wave = np.cos(2*np.pi * np.arange(N)/ (N - 1))
    hann = 0.5 * (1 - wave)
    output = np.zeros( len(input)+ N ) 


    #going through frame by frame
    for i in range (0, len(input) - N, H):
            start = i
            end = i + N


            editFrame = input [start : end] * hann

            newSpec = np.fft.fft(editFrame)

            shiftSpec = np.zeros(N, dtype=complex)

            for k in range(N // 2):
                
                newK = int(round(k * ratio))
                if ((0 < newK) & (newK < N // 2)):
                    shiftSpec[newK] = shiftSpec[newK] + newSpec[k]       
                    shiftSpec[N - newK] = shiftSpec[N - newK] + newSpec[N - k]

            outputFrame = np.real(np.fft.ifft(shiftSpec))
            
            hannFrame = outputFrame * hann        
            output[start : end] = output[start : end] + hannFrame  
    return output

In [42]:
def main():
    fs = 44100
    t = np.arange(fs) / fs
    audio = (np.sin(2 * np.pi * t) * 16000)
    num_samples = len(audio)

    #gain test
    gain = 1.5
    output = np.zeros(num_samples)
    num_blocks = num_samples // 1024
    for b in range(num_blocks):
        offset = b * 1024
        volume_control(audio[offset:offset+1024], output[offset:offset+1024], gain)
    rem = num_samples % 1024 # in case there are any leftover bytes remaining
    if rem > 0:
        offset = num_blocks * 1024
        for i in range(offset, offset+rem):
            output[i] = audio[i] * gain

    print(f"Gain Test (gain={gain})")
    print(f"input range: [{min(audio)}, {max(audio)}]")
    print(f"output range: [{min(output)}, {max(output)}]")
    
    #echo test
    delay_samples = int(0.3 * fs)  # 300ms delay
    decay = 0.5
    delay_buffer = np.zeros(delay_samples)
    echo_output = np.zeros(num_samples)

    for b in range(num_blocks):
        offset = b * 1024
        echo_control(audio[offset:offset+1024], echo_output[offset:offset+1024], delay_buffer, decay)

    print(f"Echo Test (delay = {delay_samples} samples, decay = {decay})")
    print(f"input range:  [{min(audio)}, {max(audio)}]")
    print(f"output range: [{min(echo_output)}, {max(echo_output)}]")    

    #transcription test
    result = pitch_shift(audio, tones=3)
    print("Input length:", len(audio))
    print("Output length:", len(result))
    print("First 5:", result[0:5])

    

In [43]:
if __name__ == "__main__":
    main()

Gain Test (gain=1.5)
input range: [-16000.0, 16000.0]
output range: [-24000.0, 24000.0]
Echo Test (delay = 13230 samples, decay = 0.5)
input range:  [-16000.0, 16000.0]
output range: [-23943.27241539632, 23943.27241539632]
Input length: 44100
Output length: 45124
First 5: [ 0.         -0.00548402 -0.02192785 -0.04931959 -0.08764786]
